In [1]:
import pandas as pd
import os
from pathlib import Path

### Read Sales Files

In [2]:
sales_folder = Path("raw_data")

sales_files = list(sales_folder.glob("*Sales.xlsx"))

sales_data = []

for file in sales_files:
    df = pd.read_excel(file)
    sales_data.append(df)

sales = pd.concat(sales_data, ignore_index=True)
sales.head()

,Date,Invoice No,Customer,Product,Category,Quantity,Unit Price,Unit Cost
0,2026-02-13,INV011,Ade Stores,Rice 50kg,Foodstuff,5,50000,35000
1,2026-02-14,INV012,Bello Mart,Vegetable Oil,Foodstuff,12,20000,10000
2,2026-02-15,INV013,City Restaurant,Rice 50kg,Foodstuff,8,50000,35000
3,2026-02-16,INV014,Al-Hayat Shop,Sugar 25kg,Foodstuff,12,30000,20000
4,2026-02-17,INV015,Ade Stores,Flour 50kg,Foodstuff,3,35000,25000


#### Read Expense Files

In [3]:
expense_files = list(sales_folder.glob("*Expenses.xlsx"))

expense_data = []

for file in expense_files:
    df = pd.read_excel(file)
    expense_data.append(df)

expenses = pd.concat(expense_data, ignore_index=True)

In [4]:
expenses.head()

,Date,Expense ID,Expense Category,Description,Amount
0,2026-02-02,EXP013,Transport,delivery,85000
1,2026-02-03,EXP014,Utilities,Electricity,45000
2,2026-02-04,EXP015,Salaries,Staff salaries,250000
3,2026-02-05,EXP016,Rent,Office rent,150000
4,2026-02-06,EXP017,Marketing,Online ads,35000


#### Check Metadata

In [5]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Date        30 non-null     datetime64[ns]
 1   Invoice No  30 non-null     object        
 2   Customer    30 non-null     object        
 3   Product     30 non-null     object        
 4   Category    30 non-null     object        
 5   Quantity    30 non-null     int64         
 6   Unit Price  30 non-null     int64         
 7   Unit Cost   30 non-null     int64         
dtypes: datetime64[ns](1), int64(3), object(4)
memory usage: 2.0+ KB


In [6]:
expenses.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33 entries, 0 to 32
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Date              33 non-null     datetime64[ns]
 1   Expense ID        33 non-null     object        
 2   Expense Category  33 non-null     object        
 3   Description       33 non-null     object        
 4   Amount            33 non-null     int64         
dtypes: datetime64[ns](1), int64(1), object(3)
memory usage: 1.4+ KB


#### Data Cleaning/Processing

In [7]:
sales.isnull().sum()

Date          0
Invoice No    0
Customer      0
Product       0
Category      0
Quantity      0
Unit Price    0
Unit Cost     0
dtype: int64

In [8]:
expenses.isnull().sum()

Date                0
Expense ID          0
Expense Category    0
Description         0
Amount              0
dtype: int64

In [9]:
sales["Customer"] = sales["Customer"].str.strip()
sales["Product"] = sales["Product"].str.strip()
sales["Category"] = sales["Category"].str.strip()

#### Sales Analysis

In [10]:
sales["Revenue"] = sales["Quantity"] * sales["Unit Price"]
sales["COGS"] = sales["Quantity"] * sales["Unit Cost"]
sales["Gross_Profit"] = sales["Revenue"] - sales["COGS"]
sales["Gross_Margin"] = (
    sales["Gross_Profit"] / sales["Revenue"]
) * 100

In [11]:
sales.head()

,Date,Invoice No,Customer,Product,Category,Quantity,Unit Price,Unit Cost,Revenue,COGS,Gross_Profit,Gross_Margin
0,2026-02-13,INV011,Ade Stores,Rice 50kg,Foodstuff,5,50000,35000,250000,175000,75000,30.000000
1,2026-02-14,INV012,Bello Mart,Vegetable Oil,Foodstuff,12,20000,10000,240000,120000,120000,50.000000
2,2026-02-15,INV013,City Restaurant,Rice 50kg,Foodstuff,8,50000,35000,400000,280000,120000,30.000000
3,2026-02-16,INV014,Al-Hayat Shop,Sugar 25kg,Foodstuff,12,30000,20000,360000,240000,120000,33.333333
4,2026-02-17,INV015,Ade Stores,Flour 50kg,Foodstuff,3,35000,25000,105000,75000,30000,28.571429


In [12]:
sales["Month"] = sales["Date"].dt.to_period("M").astype(str)
sales.head()

,Date,Invoice No,Customer,Product,Category,Quantity,Unit Price,Unit Cost,Revenue,COGS,Gross_Profit,Gross_Margin,Month
0,2026-02-13,INV011,Ade Stores,Rice 50kg,Foodstuff,5,50000,35000,250000,175000,75000,30.000000,2026-02
1,2026-02-14,INV012,Bello Mart,Vegetable Oil,Foodstuff,12,20000,10000,240000,120000,120000,50.000000,2026-02
2,2026-02-15,INV013,City Restaurant,Rice 50kg,Foodstuff,8,50000,35000,400000,280000,120000,30.000000,2026-02
3,2026-02-16,INV014,Al-Hayat Shop,Sugar 25kg,Foodstuff,12,30000,20000,360000,240000,120000,33.333333,2026-02
4,2026-02-17,INV015,Ade Stores,Flour 50kg,Foodstuff,3,35000,25000,105000,75000,30000,28.571429,2026-02


In [13]:
monthly_sales = sales.groupby("Month").agg(
    Revenue=("Revenue", "sum"),
    COGS=("COGS", "sum"),
    Gross_Profit=("Gross_Profit", "sum"),
    Quantity=("Quantity", "sum")
).reset_index()
monthly_sales.head()

,Month,Revenue,COGS,Gross_Profit,Quantity
0,2026-01,3085000,2045000,1040000,89
1,2026-02,3135000,2070000,1065000,90
2,2026-03,3685000,2465000,1220000,101


In [14]:
monthly_sales["Gross_Margin"] = (
    monthly_sales["Gross_Profit"] /
    monthly_sales["Revenue"]
) * 100
monthly_sales.head()

,Month,Revenue,COGS,Gross_Profit,Quantity,Gross_Margin
0,2026-01,3085000,2045000,1040000,89,33.711507
1,2026-02,3135000,2070000,1065000,90,33.971292
2,2026-03,3685000,2465000,1220000,101,33.107191


#### Product Analysis

In [15]:
product_analysis = sales.groupby("Product").agg(
    Quantity=("Quantity", "sum"),
    Revenue=("Revenue", "sum"),
    Gross_Profit=("Gross_Profit", "sum")
).reset_index()

product_analysis = product_analysis.sort_values(
    "Revenue",
    ascending=False
)
product_analysis.head()

,Product,Quantity,Revenue,Gross_Profit
1,Rice 50kg,105,5250000,1575000
2,Sugar 25kg,63,1890000,630000
3,Vegetable Oil,77,1540000,770000
0,Flour 50kg,35,1225000,350000


#### Customer Analysis

In [16]:
customer_analysis = sales.groupby("Customer").agg(
    Orders=("Invoice No", "nunique"),
    Revenue=("Revenue", "sum"),
    Gross_Profit=("Gross_Profit", "sum")
).reset_index()

customer_analysis = customer_analysis.sort_values(
    "Revenue",
    ascending=False
)

customer_analysis.head()

,Customer,Orders,Revenue,Gross_Profit
3,City Restaurant,6,2200000,840000
6,Suru Dishes,3,2100000,630000
2,Bello Mart,6,1590000,605000
0,Ade Stores,6,1495000,440000
1,Al-Hayat Shop,3,1080000,360000


#### Expenses Analysis

In [17]:
expenses["Month"] = (
    expenses["Date"]
    .dt.to_period("M")
    .astype(str)
)

In [18]:
expenses.head()

,Date,Expense ID,Expense Category,Description,Amount,Month
0,2026-02-02,EXP013,Transport,delivery,85000,2026-02
1,2026-02-03,EXP014,Utilities,Electricity,45000,2026-02
2,2026-02-04,EXP015,Salaries,Staff salaries,250000,2026-02
3,2026-02-05,EXP016,Rent,Office rent,150000,2026-02
4,2026-02-06,EXP017,Marketing,Online ads,35000,2026-02


In [19]:
monthly_expenses = expenses.groupby("Month").agg(
    Operating_Expenses=("Amount", "sum")
).reset_index()
monthly_expenses.head()

,Month,Operating_Expenses
0,2026-01,900000
1,2026-02,900000
2,2026-03,900000


#### Financial Summary

In [20]:
financial_summary = monthly_sales.merge(
    monthly_expenses,
    on="Month",
    how="left"
)
financial_summary.head()

,Month,Revenue,COGS,Gross_Profit,Quantity,Gross_Margin,Operating_Expenses
0,2026-01,3085000,2045000,1040000,89,33.711507,900000
1,2026-02,3135000,2070000,1065000,90,33.971292,900000
2,2026-03,3685000,2465000,1220000,101,33.107191,900000


In [21]:
financial_summary["Net_Profit"] = (
    financial_summary["Gross_Profit"] -
    financial_summary["Operating_Expenses"]
)

financial_summary["Net_Margin"] = (
    financial_summary["Net_Profit"] /
    financial_summary["Revenue"]
) * 100

financial_summary.head()

,Month,Revenue,COGS,Gross_Profit,Quantity,Gross_Margin,Operating_Expenses,Net_Profit,Net_Margin
0,2026-01,3085000,2045000,1040000,89,33.711507,900000,140000,4.538088
1,2026-02,3135000,2070000,1065000,90,33.971292,900000,165000,5.263158
2,2026-03,3685000,2465000,1220000,101,33.107191,900000,320000,8.683853


#### Management Report

In [22]:
output_file = "reports/Automated_Management_Report.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    sales.to_excel(
        writer,
        sheet_name="Sales Data",
        index=False
    )

    expenses.to_excel(
        writer,
        sheet_name="Expenses",
        index=False
    )

    financial_summary.to_excel(
        writer,
        sheet_name="Financial Summary",
        index=False
    )

    product_analysis.to_excel(
        writer,
        sheet_name="Product Analysis",
        index=False
    )

    customer_analysis.to_excel(
        writer,
        sheet_name="Customer Analysis",
        index=False
    )

#### Format Financial Summary Sheet

In [23]:
from openpyxl import load_workbook

wb = load_workbook(output_file)

ws = wb["Financial Summary"]

ws.freeze_panes = "A2"

for column in ws.columns:
    max_length = 0
    column_letter = column[0].column_letter

    for cell in column:
        if cell.value:
            max_length = max(
                max_length,
                len(str(cell.value))
            )

    ws.column_dimensions[column_letter].width = max_length + 2

wb.save(output_file)